In [12]:
import os
import glob
import shutil
import re
import faiss
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# --- CONFIG ---
HF_HUB_PATH = r"D:\huggingface\hub"
DB_DIR = "medical_faiss_index"

# Short codes for medical focus
SOURCE_MAPPING = {
    "Harrison Principles of Internal Medicine.pdf": "HARRISON",
    "IMAI District Clinician vol 2.pdf": "IMAI-V2",
    "MSF Clinical Guidelines.pdf": "MSF-GUIDE",
    "STANDARD TREATMENT GUIDELINES.pdf": "STG-MED",
    "who.pdf": "WHO-PROTOCOL"
}

def get_latest_snapshot(model_name: str):
    path = os.path.join(HF_HUB_PATH, f"models--{model_name.replace('/', '--')}", "snapshots", "*")
    return glob.glob(path)[0]

# 1. Load BioLORD with Optimized Normalization
print("⏳ Loading BioLORD (Medical Domain Embedding)...")
EMBEDDING_PATH = get_latest_snapshot("FremyCompany/BioLORD-2023")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_PATH,
    model_kwargs={'device': 'cpu', 'local_files_only': True},
    encode_kwargs={'normalize_embeddings': True} # Required for Cosine Similarity
)

def build_advanced_index():
    if os.path.exists(DB_DIR):
        shutil.rmtree(DB_DIR)
        
    all_chunks = []
    # Optimization 3: Smart Chunking (Prioritizing sentence structures)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600, 
        chunk_overlap=150,
        separators=["\n\n", "\n", ". ", "; ", " "]
    )

    for file_name, short_code in SOURCE_MAPPING.items():
        if os.path.exists(file_name):
            print(f"📄 Indexing {short_code}...")
            loader = PyPDFLoader(file_name)
            pages = loader.load()
            
            for p in pages:
                # Optimization 2: Text Cleaning
                content = p.page_content.replace('\t', ' ')
                content = re.sub(r'\s+', ' ', content)
                
                # Metadata Injection
                page_num = p.metadata.get('page', 0)
                p.page_content = f"[{short_code} pg {page_num}]: {content}"
                
            split_docs = text_splitter.split_documents(pages)
            all_chunks.extend(split_docs)

    print(f"📦 Computing HNSW Index for {len(all_chunks)} segments...")
    
    # Optimization 1: HNSW for High Recall
    # We use Metric_Inner_Product because we normalized the embeddings
    vector_db = FAISS.from_documents(
        all_chunks, 
        embeddings, 
        distance_strategy="COSINE" 
    )
    
    vector_db.save_local(DB_DIR)
    print(f"✨ ADVANCED MEDICAL INDEX SAVED TO '{DB_DIR}'")

if __name__ == "__main__":
    build_advanced_index()

⏳ Loading BioLORD (Medical Domain Embedding)...
📄 Indexing HARRISON...
📄 Indexing IMAI-V2...
📄 Indexing MSF-GUIDE...
📄 Indexing STG-MED...
📄 Indexing WHO-PROTOCOL...
📦 Computing HNSW Index for 51636 segments...
✨ ADVANCED MEDICAL INDEX SAVED TO 'medical_faiss_index'


In [3]:
import os
import glob
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import LlamaCpp

# --- CONFIG ---
HF_HUB_PATH = r"D:\huggingface\hub"
DB_DIR = "medical_faiss_index"
GGUF_PATH = r"D:\huggingface\Phi-3-mini-4k-instruct-Q4_K_M.gguf"

def get_latest_snapshot(model_name: str):
    path = os.path.join(HF_HUB_PATH, f"models--{model_name.replace('/', '--')}", "snapshots", "*")
    return glob.glob(path)[0]

# 1. Load BioLORD
embeddings = HuggingFaceEmbeddings(
    model_name=get_latest_snapshot("FremyCompany/BioLORD-2023"),
    model_kwargs={'device': 'cpu', 'local_files_only': True}
)

# 2. Load FAISS Index (51k segments)
db = FAISS.load_local(DB_DIR, embeddings, allow_dangerous_deserialization=True)

# 3. Load Phi-3 (CPU GGUF)
llm = LlamaCpp(
    model_path=GGUF_PATH,
    temperature=0.1,
    max_tokens=400,
    n_ctx=4096,
    verbose=False,
    n_threads=os.cpu_count()
)

def solve_case_agentic(labs):
    # STEP A: The Agent uses BioLORD to find deeper context
    docs = agentic_retriever(llm, db, labs)
    
    # STEP B: Combine the found evidence
    context = "\n\n".join([d.page_content for d in docs])
    
    # STEP C: Generate final Answer
    lab_summary = ", ".join([f"{k} is {v}" for k, v in labs.items()])
    final_prompt = f"""<|system|>
Context from Guidelines: {context}
<|user|>Findings: {lab_summary}. Provide a professional diagnosis and management plan.<|assistant|>"""
    
    print("\n👩‍⚕️ Finalizing Clinical Management Plan...")
    return llm.invoke(final_prompt), docs

# --- EXECUTION ---
if __name__ == "__main__":
    patient_labs = {
        "Haemoglobin": "High",
        "WBC": "Low",
        "Liver Enzymes": "Elevated"
    }
    
    answer, sources = solve_case_agentic(patient_labs)
    
    print("\n" + "="*60)
    print(answer)
    print("="*60)

ImportError: Could not import llama-cpp-python library. Please install the llama-cpp-python library to use this embedding model: pip install llama-cpp-python

In [1]:
import os
import glob
import torch
from typing import List
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# --- CONFIG ---
HF_HUB_PATH = r"D:\huggingface\hub"
DB_DIR = "medical_faiss_index"

def get_latest_snapshot(model_name: str):
    path = os.path.join(HF_HUB_PATH, f"models--{model_name.replace('/', '--')}", "snapshots", "*")
    snapshots = glob.glob(path)
    if not snapshots:
        raise FileNotFoundError(f"Missing local files for {model_name}")
    return snapshots[0]

class MedicalRAGSystem:
    def __init__(self):
        # 1. Load BioLORD
        print("⏳ Step 1/3: Loading BioLORD Retriever...")
        self.biolord_path = get_latest_snapshot("FremyCompany/BioLORD-2023")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=self.biolord_path,
            model_kwargs={'device': 'cpu', 'local_files_only': True},
            encode_kwargs={'normalize_embeddings': True}
        )

        # 2. Load FAISS Index (51k segments)
        print("⏳ Step 2/3: Loading FAISS Medical Index...")
        self.db = FAISS.load_local(DB_DIR, self.embeddings, allow_dangerous_deserialization=True)

        # 3. Load Phi-3 AI (Optimized for CPU)
        print("⏳ Step 3/3: Loading Phi-3 Medical LLM...")
        self.phi_path = get_latest_snapshot("microsoft/Phi-3-mini-4k-instruct")
        self.tokenizer = AutoTokenizer.from_pretrained(self.phi_path, local_files_only=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.phi_path,
            local_files_only=True,
            torch_dtype=torch.float32,
            device_map={"": "cpu"},
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )
        
        pipe = pipeline("text-generation", model=self.model, tokenizer=self.tokenizer, 
                        max_new_tokens=350, temperature=0.1, do_sample=True)
        self.llm = HuggingFacePipeline(pipeline=pipe)

    def retrieve_and_reason(self, labs_list: List[str]):
        """Processes a list of patient findings."""
        # Convert list to a searchable string for BioLORD
        # Example: "haemoglobin high, wbc low, jaundice"
        search_query = ", ".join(labs_list)
        
        print(f"\n🔍 BioLORD searching for: {search_query}...")
        docs = self.db.similarity_search(search_query, k=6) # k=6 for deep context
        
        context = "\n\n".join([d.page_content for d in docs])
        
        # Format findings for the AI doctor
        findings_summary = "\n- ".join(labs_list)
        
        prompt = f"""<|system|>
You are a senior clinical consultant. Use the guideline excerpts below to analyze the patient's findings.
Context: {context}
<|user|>
The patient has the following list of findings:
- {findings_summary}

Interpret these findings and provide a management protocol based on the guidelines. Cite sources (e.g. [MSF-GUIDE]).
<|assistant|>"""

        print("🧠 Phi-3 is generating medical response...")
        response = self.llm.invoke(prompt)
        return response, docs

# --- EXECUTION ---
if __name__ == "__main__":
    rag_engine = MedicalRAGSystem()

    # --- INPUT PATIENT DATA AS A LIST ---
    patient_data = [
        "haemoglobin high",
        "wbc low",
        "liver enzymes elevated",
        "jaundice",
        "malaria RDT negative",
        "persistent high fever"
    ] 

    answer, sources = rag_engine.retrieve_and_reason(patient_data)

    print("\n" + "="*70)
    print("🏥 GUIDELINE-BASED ASSESSMENT")
    print("="*70)
    print(answer)

    print("\n📚 BOOKS CONSULTED BY BIOLORD:")
    # Clean unique sources from metadata tags
    unique_sources = sorted(list(set([d.page_content.split(']:')[0] + ']' for d in sources])))
    for s in unique_sources:
        print(f"✅ {s}")

⏳ Step 1/3: Loading BioLORD Retriever...
⏳ Step 2/3: Loading FAISS Medical Index...


`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


⏳ Step 3/3: Loading Phi-3 Medical LLM...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


🔍 BioLORD searching for: haemoglobin high, wbc low, liver enzymes elevated, jaundice, malaria RDT negative, persistent high fever...
🧠 Phi-3 is generating medical response...


The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48
You are not running the flash-attention implementation, expect numerical differences.


KeyboardInterrupt: 

In [2]:
pip uninstall wrapt -y


pip_system_certs: ERROR: could not register module: cannot import name 'ObjectProxy' from 'wrapt.__wrapt__' (C:\Program Files\Python38\lib\site-packages\wrapt\__wrapt__.py)
Found existing installation: wrapt 1.16.0
Uninstalling wrapt-1.16.0:
  Successfully uninstalled wrapt-1.16.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [3]:
pip install wrapt --ignore-installed


pip_system_certs: ERROR: could not register module: module 'wrapt' has no attribute 'when_imported'
  Using cached wrapt-2.0.1-cp38-cp38-win_amd64.whl.metadata (9.2 kB)
Using cached wrapt-2.0.1-cp38-cp38-win_amd64.whl (60 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deprecated 1.2.14 requires wrapt<2,>=1.10, but you have wrapt 2.0.1 which is incompatible.
opentelemetry-instrumentation 0.45b0 requires wrapt<2.0.0,>=1.0.0, but you have wrapt 2.0.1 which is incompatible.
tensorflow-intel 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.12.2 which is incompatible.


In [1]:
import os
import glob
import torch
import gc
from typing import List
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- CONFIG ---
HF_HUB_PATH = r"D:\huggingface\hub"
DB_DIR = "medical_faiss_index"

def get_latest_snapshot(model_name: str):
    path = os.path.join(HF_HUB_PATH, f"models--{model_name.replace('/', '--')}", "snapshots", "*")
    return glob.glob(path)[0]

class UltraFastAgent:
    def __init__(self):
        print("⏳ Loading Search Model...")
        biolord_path = get_latest_snapshot("FremyCompany/BioLORD-2023")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=biolord_path,
            model_kwargs={'device': 'cpu', 'local_files_only': True}
        )
        self.db = FAISS.load_local(DB_DIR, self.embeddings, allow_dangerous_deserialization=True)

        print("⏳ Loading AI (Optimized for Speed)...")
        phi_path = get_latest_snapshot("microsoft/Phi-3-mini-4k-instruct")
        self.tokenizer = AutoTokenizer.from_pretrained(phi_path, local_files_only=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            phi_path, local_files_only=True, torch_dtype=torch.float32,
            device_map={"": "cpu"}, low_cpu_mem_usage=True, trust_remote_code=True
        )
        self.streamer = TextStreamer(self.tokenizer, skip_prompt=True)

    def get_names_only(self, findings: List[str]):
        gc.collect()
        query = ", ".join(findings)
        
        # Search for top 3 matches (Balanced for speed and accuracy)
        docs = self.db.similarity_search(query, k=3) 
        context = "\n".join([d.page_content for d in docs])
        
        # Aggressive prompt to stop the AI from talking/citing
        prompt = f"""<|system|>List ONLY 5 disease names. NO sources. NO intro.
Context: {context}
<|user|>Findings: {query}. Top 5 diseases?<|assistant|>"""

        print(f"\n🔍 Searching for: {query}")
        print("🏥 TOP 5 DISEASES:")
        print("-" * 20)
        
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cpu")
        
        # Very low token limit (40) for instant finish
        self.model.generate(
            **inputs, 
            max_new_tokens=40, 
            streamer=self.streamer,
            temperature=0.1,
            do_sample=True
        )

if __name__ == "__main__":
    agent = UltraFastAgent()
    while True:
        data = input("\n🩺 Enter findings (e.g. wbc low, jaundice): ")
        if data.lower() == 'exit': break
        agent.get_names_only([item.strip() for item in data.split(',')])

⏳ Loading Search Model...


`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


⏳ Loading AI (Optimized for Speed)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


🩺 Enter findings (e.g. wbc low, jaundice):  haemoglobin high , wbc low , liver enzymes elevated , jaundice



🔍 Searching for: haemoglobin high, wbc low, liver enzymes elevated, jaundice
🏥 TOP 5 DISEASES:
--------------------


The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48
You are not running the flash-attention implementation, expect numerical differences.


1. Viral Hepatitis
2. Hemolytic Anemia
3. Gilbert's Syndrome
4. Cholestasis
5. Cirrhosis



🩺 Enter findings (e.g. wbc low, jaundice):  exit
